In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import plotly.express as px

# ============================================================
# Config
# ============================================================
BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "event_representation_matrix_window0_scaled.csv"

OUT_DIR = BASE_DIR / "event_clustering_outputs"
OUT_DIR.mkdir(exist_ok=True)

N_CLUSTERS = 3
RANDOM_STATE = 42

# ============================================================
# 1. Load
# ============================================================
df = pd.read_csv(INPUT_FILE)
df["Date"] = pd.to_datetime(df["Date"])

feature_cols = [col for col in df.columns if col != "Date"]
X = df[feature_cols].values

print(f"[INFO] N events: {len(df)}")
print(f"[INFO] Features: {feature_cols}")

# ============================================================
# 2. KMeans Clustering
# ============================================================
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=20
)

labels = kmeans.fit_predict(X)
df["Cluster"] = labels.astype(str)   # plotly color 구분용 문자열 변환

# ============================================================
# 3. Cluster Centroid
# ============================================================
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=feature_cols
)
centroids["Cluster"] = range(N_CLUSTERS)

# ============================================================
# 4. PCA (2D 시각화용)
# ============================================================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

df["PC1"] = X_pca[:, 0]
df["PC2"] = X_pca[:, 1]

# hover에 보기 좋게 날짜 문자열 추가
df["Date_str"] = df["Date"].dt.strftime("%Y-%m-%d")

# ============================================================
# 5. Save
# ============================================================
df.to_csv(OUT_DIR / "event_cluster_result.csv", index=False)
centroids.to_csv(OUT_DIR / "event_cluster_centroids.csv", index=False)

df[["Date", "Cluster", "PC1", "PC2"]].to_csv(
    OUT_DIR / "event_cluster_pca.csv", index=False
)

# ============================================================
# 6. Interactive Plot (Plotly)
# ============================================================
fig = px.scatter(
    df,
    x="PC1",
    y="PC2",
    color="Cluster",
    hover_data={
        "Date_str": True,
        "Cluster": True,
        "PC1": ':.4f',
        "PC2": ':.4f',
        "Date": False
    },
    title="Event Clustering (PCA)",
    width=900,
    height=700
)

fig.update_traces(
    marker=dict(size=10),
    selector=dict(mode="markers")
)

fig.update_layout(
    xaxis_title="PC1",
    yaxis_title="PC2",
    legend_title="Cluster",
    template="plotly_white"
)

# HTML 저장
fig.write_html(OUT_DIR / "event_cluster_pca_interactive.html")

print("[INFO] Clustering done")
print(f"[INFO] Interactive plot saved to: {OUT_DIR / 'event_cluster_pca_interactive.html'}")

# VS Code / Jupyter에서 바로 표시
fig.show()

[INFO] N events: 38
[INFO] Features: ['Magnitude', 'Top1_DeltaS', 'Top2_DeltaS', 'Top3_DeltaS', 'Peak', 'PeakHorizon', 'HalfLife', 'DurationAboveHalf', 'AUC']
[INFO] Clustering done
[INFO] Interactive plot saved to: event_clustering_outputs\event_cluster_pca_interactive.html
